# Aaru Data Integration — Practice Take-Home Simulation

**This is a practice exercise simulating what a Data Integration take-home at Aaru would plausibly look like**, based on:
- The JD's explicit emphasis on entity resolution, schema harmonization, and data quality at scale
- Aaru's real domain (demographic/panel/behavioral data feeding population simulations)
- The structure of the Woodline Partners take-home you already mastered (same underlying skill: messy multi-source data → resolved, trusted, joined dataset)

**Scenario:** You've been given two data sources describing an overlapping population — a demographic panel vendor export (`panel_data.csv`) and a survey response export (`survey_data.csv`). Your job: resolve entities across the two sources, harmonize schema, document every data quality issue you find, and produce one clean, joined, analysis-ready dataset.

**Treat this like the real thing: set a timer for 2.5–3 hours and work through it without peeking at the hints unless you're stuck for 5+ minutes.**

---


## Section 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

panel = pd.read_csv('panel_data.csv')
survey = pd.read_csv('survey_data.csv')

print("PANEL:", panel.shape)
print("SURVEY:", survey.shape)


## Section 2: Inspection

**Your task:** Before writing any transformation code, inspect both datasets thoroughly. What you're looking for:
- Column names, dtypes
- Nulls per column
- Duplicate rows
- Range/sanity issues (impossible values)
- Format inconsistencies in fields that should represent the same concept across sources (names, dates, income)

<details><summary>💡 Hint 1 (click to expand)</summary>
Write a small `quick_eda(df, name)` helper you can call on both dataframes — shape, dtypes, null counts, and `.head()`. Don't skip printing the actual rows; the messiness here is mostly visible by eye, not by `.describe()`.
</details>


In [ ]:
# YOUR CODE HERE
def quick_eda(df, name):
    print(f"=== {name} ===")
    print(df.dtypes)
    print("\nNulls:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    print("\n", df.head(8))
    print()

quick_eda(panel, "PANEL")
quick_eda(survey, "SURVEY")


## Section 3: Data Quality Log

**Your task:** Build a running, structured log of every data quality issue you find. This is one of the things graders look at most closely — not just *that* you found issues, but whether you documented them clearly enough that someone else could act on your findings.

By now you should have spotted (at minimum):
- Two different date formats across sources
- Two different name formats across sources (plus casing inconsistencies)
- Income represented two different ways (numeric estimate vs. band)
- Some missing zip codes
- Possible duplicate panel records

<details><summary>💡 Hint 2</summary>
Structure your log as a list of dicts: `{'field': ..., 'issue': ..., 'severity': ..., 'recommended_fix': ...}`. Print it as a dataframe at the end — it reads like a deliverable, not a debugging scratchpad.
</details>


In [ ]:
# YOUR CODE HERE
quality_log = []

def log_issue(field, issue, severity, fix):
    quality_log.append({'field': field, 'issue': issue, 'severity': severity, 'recommended_fix': fix})

# Example to get you started -- add the rest yourself:
log_issue(
    field='panel_join_date / response_date',
    issue='Mixed date formats across sources (ISO YYYY-MM-DD in panel, MM/DD/YYYY in survey)',
    severity='Medium',
    fix='Parse both explicitly with known format strings before any date arithmetic or sorting'
)

# TODO: add entries for name format mismatch, income representation mismatch,
# missing zips, duplicate panel records, imperfect population overlap


## Section 4: Schema Harmonization

**Your task:** Before you can join the two sources, get them into a shared schema:
- Parse both date fields into actual `datetime` objects
- Split or recombine name fields into a consistent format (recommend: separate `first`/`last` columns on both sides)
- Reconcile the income representation — panel gives a numeric estimate, survey gives a band. You'll need to bucket the panel's numeric estimate into the same bands survey uses (or vice versa) before you can compare them meaningfully.
- Standardize casing (e.g., title-case all names) so format differences don't get mistaken for entity differences in the next step

<details><summary>💡 Hint 3 — name splitting</summary>
Panel format is `"Last, First"` (sometimes abbreviated or all-caps). Survey format is `"First Last"`. Don't assume the comma is always present after you uppercase/lowercase — strip and `.title()` BEFORE you split, not after, or "JACKSON, ROBERT" and "Jackson, Robert" will parse inconsistently if your split logic is case-sensitive.
</details>

<details><summary>💡 Hint 4 — income bucketing</summary>
Define the band boundaries once as a function, e.g. `bucket_income(x)` returning one of `<35k / 35-65k / 65-100k / 100-150k / 150k+`, and apply it to the panel's numeric column. Apply the SAME function logic everywhere bands are derived, so panel-derived bands and survey-native bands use identical boundaries.
</details>


In [ ]:
# YOUR CODE HERE

def split_panel_name(name):
    # "Last, First" or "LAST, FIRST" or "Last, F." -> (first, last), title-cased
    name = name.strip()
    last, first = [p.strip() for p in name.split(',', 1)]
    return first.title(), last.title()

panel[['first_clean', 'last_clean']] = panel['panelist_name'].apply(
    lambda x: pd.Series(split_panel_name(x))
)

def split_survey_name(name):
    parts = name.strip().split(' ', 1)
    return parts[0].title(), parts[1].title()

survey[['first_clean', 'last_clean']] = survey['respondent_full_name'].apply(
    lambda x: pd.Series(split_survey_name(x))
)

# TODO: parse dates explicitly (pd.to_datetime with format=...)
# TODO: write bucket_income() and apply to panel's est_annual_income_usd
# TODO: standardize zip as string, not float (watch for the missing-value-induced float cast!)


## Section 5: Entity Resolution

**Your task:** Build a canonical entity key that lets you match the same person across panel and survey, given that:
- Names are now harmonized in format but may still have minor typos/variants in a real dataset (this synthetic version is mostly clean post-harmonization, but **say out loud in your summary** how you'd handle fuzzy variants in a real, larger version of this problem)
- Not every panelist is in the survey, and not every survey respondent is in the panel — your matching logic needs to handle non-matches gracefully, not assume a 1:1 join

<details><summary>💡 Hint 5 — blocking</summary>
Even at this small scale, demonstrate the *pattern* you'd use at real scale: block candidate matches by `last_clean` (cheap key) before comparing `first_clean` + any other available field, rather than a naive full cross-join. At >100TB this is the difference between tractable and not.
</details>

<details><summary>💡 Hint 6 — match confidence</summary>
Don't just inner-join and call it done. Tag each match with a confidence level: exact first+last match = high confidence; first+last match but age differs by >2 years = flag for review; no match found in either direction = log as unmatched, don't silently drop.
</details>


In [ ]:
# YOUR CODE HERE

# Build blocking key
panel['block_key'] = panel['last_clean'].str.lower()
survey['block_key'] = survey['last_clean'].str.lower()

# TODO: within each block, match on first_clean (+ age proximity as a tiebreaker/confidence signal)
# TODO: produce a match table: panel_idx, survey_idx, match_confidence
# TODO: explicitly count and report: how many panelists matched, how many didn't,
#       how many survey respondents have no panel match at all


## Section 6: Integration & Validation

**Your task:** Produce one final, joined, analysis-ready dataframe. Then validate it:
- Row count sanity: does the final row count make sense given what you found in entity resolution?
- No silent data loss: anyone dropped should be explainable by something you logged, not a mystery
- Spot-check a few resolved records by eye

<details><summary>💡 Hint 7</summary>
Print three things explicitly at the end: (1) the quality_log as a table, (2) match rate (% of panel matched to survey, % of survey matched to panel), (3) a short written summary of what you'd verify with more time or access to the source teams. Graders weight the documentation almost as much as the code.
</details>


In [ ]:
# YOUR CODE HERE

# TODO: build final merged dataframe from your match table
# TODO: print quality_log as pd.DataFrame(quality_log)
# TODO: print match rate summary
# TODO: write 3-5 sentences: what you'd verify next, what you'd automate if this
#       were a recurring pipeline rather than a one-time exercise


## Section 7: Self-Grade Checklist

Before you consider this "done," check yourself against what a real grader (e.g., someone like Dan Kenefick at Woodline, or Aaru's data team) would look for:

- [ ] Did I inspect both raw files before writing any transformation code?
- [ ] Is every data quality issue logged with field, issue, severity, and a recommended fix — not just mentioned in passing?
- [ ] Did I harmonize schema (dates, names, income bands) using explicit, reusable logic rather than one-off string hacks?
- [ ] Did I use a blocking strategy for entity resolution rather than a naive full cross-product, and would I say *why* out loud if asked about scale?
- [ ] Did I tag match confidence rather than treating every match as equally certain?
- [ ] Did I explicitly report unmatched records on both sides rather than silently dropping them?
- [ ] Did I validate the final dataset (row counts, spot checks) rather than assuming the join worked?
- [ ] Did I write a short, clear summary of limitations and next steps?

If you can check all eight honestly, you're at the level this take-home is designed to test for.
